In [93]:
import csv
import json
from datetime import datetime

CAMINHO_ARQUIVO = "/content/drive/MyDrive/MANIPULAÇAO DE DADOS/Transacoes.csv"

def ler_transacoes(caminho_arquivo):
    transacoes = []

    try:
        with open(caminho_arquivo, mode="r", encoding="utf-8-sig") as arquivo:
            leitor = csv.DictReader(arquivo, delimiter=";")

            for linha in leitor:
                transacoes.append(linha)

    except FileNotFoundError:
        print(f"Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")


    return transacoes

In [94]:
transacoes = ler_transacoes(CAMINHO_ARQUIVO)

for transacao in transacoes:
    print(transacao)

{'id': '1', 'data': '05/01/2026', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500', 'descricao': 'Salário janeiro', 'categoria': 'salario'}
{'id': '2', 'data': '08/01/2026', 'cliente_id': 'CLI002', 'tipo': 'debito', 'valor': '180.5', 'descricao': 'Supermercado', 'categoria': 'compra'}
{'id': '3', 'data': '12/01/2026', 'cliente_id': 'CLI003', 'tipo': 'credito', 'valor': '4200', 'descricao': 'Pagamento projeto', 'categoria': 'servico'}
{'id': '4', 'data': '20/01/2026', 'cliente_id': 'CLI001', 'tipo': 'debito', 'valor': '850', 'descricao': 'Aluguel', 'categoria': 'moradia'}
{'id': '5', 'data': '25/01/2026', 'cliente_id': 'CLI004', 'tipo': 'debito', 'valor': '120.9', 'descricao': 'Farmácia', 'categoria': 'saude'}
{'id': '6', 'data': '03/02/2026', 'cliente_id': 'CLI002', 'tipo': 'credito', 'valor': '3800', 'descricao': 'Salário fevereiro', 'categoria': 'salario'}
{'id': '7', 'data': '08/02/2026', 'cliente_id': 'CLI005', 'tipo': 'debito', 'valor': '250', 'descricao': 'Internet', 'c

In [95]:
from datetime import datetime

def validar_transacao(transacao):

    try:
        # ID
        if not transacao["id"].isdigit():
            return None

        # Cliente
        if transacao["cliente_id"].strip() == "":
            return None

        # Data
        data = datetime.strptime(transacao["data"], "%d/%m/%Y")

        # Tipo
        if transacao["tipo"] not in ["credito", "debito"]:
            return None

        # Valor
        valor = float(transacao["valor"])

        if valor <= 0:
            return None

        return {
            "id": int(transacao["id"]),
            "data": data,
            "cliente_id": transacao["cliente_id"],
            "tipo": transacao["tipo"],
            "valor": valor,
            "descricao": transacao["descricao"],
            "categoria": transacao["categoria"]
        }

    except ValueError:
        return None

In [96]:
transacoes = ler_transacoes(CAMINHO_ARQUIVO)

transacoes_validas = []
transacoes_invalidas = []

for linha in transacoes:

    transacao = validar_transacao(linha)

    if transacao:
        transacoes_validas.append(transacao)
    else:
        transacoes_invalidas.append(linha)

print(f"Total de transações válidas: {len(transacoes_validas)}")
print(f"Total de transações inválidas: {len(transacoes_invalidas)}")

Total de transações válidas: 18
Total de transações inválidas: 5


In [97]:
for t in transacoes_validas:
    print(t)

{'id': 1, 'data': datetime.datetime(2026, 1, 5, 0, 0), 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': 3500.0, 'descricao': 'Salário janeiro', 'categoria': 'salario'}
{'id': 2, 'data': datetime.datetime(2026, 1, 8, 0, 0), 'cliente_id': 'CLI002', 'tipo': 'debito', 'valor': 180.5, 'descricao': 'Supermercado', 'categoria': 'compra'}
{'id': 3, 'data': datetime.datetime(2026, 1, 12, 0, 0), 'cliente_id': 'CLI003', 'tipo': 'credito', 'valor': 4200.0, 'descricao': 'Pagamento projeto', 'categoria': 'servico'}
{'id': 4, 'data': datetime.datetime(2026, 1, 20, 0, 0), 'cliente_id': 'CLI001', 'tipo': 'debito', 'valor': 850.0, 'descricao': 'Aluguel', 'categoria': 'moradia'}
{'id': 5, 'data': datetime.datetime(2026, 1, 25, 0, 0), 'cliente_id': 'CLI004', 'tipo': 'debito', 'valor': 120.9, 'descricao': 'Farmácia', 'categoria': 'saude'}
{'id': 6, 'data': datetime.datetime(2026, 2, 3, 0, 0), 'cliente_id': 'CLI002', 'tipo': 'credito', 'valor': 3800.0, 'descricao': 'Salário fevereiro', 'categoria': 'sala

In [98]:
LIMITE_SUSPEITO = 10000.00

def gerar_relatorio(transacoes):

    resumo = {}
    transacoes_suspeitas = []

    for transacao in transacoes:

        mes = transacao["data"].strftime("%Y-%m")

        if mes not in resumo:
            resumo[mes] = {
                "quantidade": 0,
                "total_credito": 0,
                "total_debito": 0,
                "saldo": 0,
                "maior_valor": 0,
                "menor_valor": float("inf"),
                "soma_valores": 0
            }

        # Quantidade de transações
        resumo[mes]["quantidade"] += 1

        # Soma de créditos e débitos
        if transacao["tipo"] == "credito":
            resumo[mes]["total_credito"] += transacao["valor"]
        elif transacao["tipo"] == "debito":
            resumo[mes]["total_debito"] += transacao["valor"]

        # Soma dos valores
        resumo[mes]["soma_valores"] += transacao["valor"]

        # Maior valor
        if transacao["valor"] > resumo[mes]["maior_valor"]:
            resumo[mes]["maior_valor"] = transacao["valor"]

        # Menor valor
        if transacao["valor"] < resumo[mes]["menor_valor"]:
            resumo[mes]["menor_valor"] = transacao["valor"]

        # Identifica transações suspeitas
        if transacao["valor"] > LIMITE_SUSPEITO:
            transacoes_suspeitas.append({
                "id": transacao["id"],
                "cliente_id": transacao["cliente_id"],
                "data": transacao["data"],
                "valor": transacao["valor"]
            })

    # Cálculos finais de cada mês
    for mes in resumo:

        resumo[mes]["saldo"] = (
            resumo[mes]["total_credito"] - resumo[mes]["total_debito"]
        )

        resumo[mes]["media"] = (
            resumo[mes]["soma_valores"] / resumo[mes]["quantidade"]
        )

    return resumo, transacoes_suspeitas

In [99]:
def calcular_periodo(transacoes):

    datas = [transacao["data"] for transacao in transacoes]

    data_inicial = min(datas)
    data_final = max(datas)

    total_dias = (data_final - data_inicial).days

    return data_inicial, data_final, total_dias

In [100]:
data_inicial, data_final, total_dias = calcular_periodo(transacoes_validas)

print(data_inicial.strftime("%d/%m/%Y"))
print(data_final.strftime("%d/%m/%Y"))
print(f"{total_dias} dias")

05/01/2026
28/03/2026
82 dias


In [101]:
def exibir_relatorio(
    relatorio,
    suspeitas,
    data_inicial,
    data_final,
    total_dias,
    total_validas,
    total_invalidas
):

    print("=" * 60)
    print("RELATÓRIO DE TRANSAÇÕES")
    print("=" * 60)

    print(f"Período analisado: {data_inicial.strftime('%d/%m/%Y')} até {data_final.strftime('%d/%m/%Y')}")
    print(f"Total de dias: {total_dias}")

    print()

    print(f"Transações válidas: {total_validas}")
    print(f"Transações inválidas: {total_invalidas}")

    print("=" * 60)

    # Relatório mensal
    for mes, dados in relatorio.items():

        print(f"\nMês: {mes}")
        print("-" * 40)

        print(f"Quantidade de transações: {dados['quantidade']}")
        print(f"Total de créditos: R$ {dados['total_credito']:.2f}")
        print(f"Total de débitos: R$ {dados['total_debito']:.2f}")
        print(f"Saldo: R$ {dados['saldo']:.2f}")
        print(f"Maior valor: R$ {dados['maior_valor']:.2f}")
        print(f"Menor valor: R$ {dados['menor_valor']:.2f}")
        print(f"Média: R$ {dados['media']:.2f}")

    print("\n" + "=" * 60)
    print("TRANSAÇÕES SUSPEITAS")
    print("=" * 60)

    if len(suspeitas) == 0:
        print("Nenhuma transação suspeita encontrada.")
    else:

        for transacao in suspeitas:

            print(f"ID: {transacao['id']}")
            print(f"Cliente: {transacao['cliente_id']}")
            print(f"Data: {transacao['data'].strftime('%d/%m/%Y')}")
            print(f"Valor: R$ {transacao['valor']:.2f}")
            print("-" * 40)

In [102]:
relatorio, suspeitas = gerar_relatorio(transacoes_validas)

In [103]:
print(type(relatorio))
print(type(suspeitas))

<class 'dict'>
<class 'list'>


In [104]:
exibir_relatorio(
    relatorio,
    suspeitas,
    data_inicial,
    data_final,
    total_dias,
    len(transacoes_validas),
    len(transacoes_invalidas)
)

RELATÓRIO DE TRANSAÇÕES
Período analisado: 05/01/2026 até 28/03/2026
Total de dias: 82

Transações válidas: 18
Transações inválidas: 5

Mês: 2026-01
----------------------------------------
Quantidade de transações: 6
Total de créditos: R$ 11800.00
Total de débitos: R$ 1151.40
Saldo: R$ 10648.60
Maior valor: R$ 4200.00
Menor valor: R$ 120.90
Média: R$ 2158.57

Mês: 2026-02
----------------------------------------
Quantidade de transações: 6
Total de créditos: R$ 18800.00
Total de débitos: R$ 1960.75
Saldo: R$ 16839.25
Maior valor: R$ 15000.00
Menor valor: R$ 250.00
Média: R$ 3460.12

Mês: 2026-03
----------------------------------------
Quantidade de transações: 6
Total de créditos: R$ 33400.00
Total de débitos: R$ 1789.99
Saldo: R$ 31610.01
Maior valor: R$ 25000.00
Menor valor: R$ 90.00
Média: R$ 5865.00

TRANSAÇÕES SUSPEITAS
ID: 8
Cliente: CLI003
Data: 10/02/2026
Valor: R$ 15000.00
----------------------------------------
ID: 13
Cliente: CLI005
Data: 10/03/2026
Valor: R$ 25000.00
---

In [105]:
def salvar_json(
    relatorio,
    total_validas,
    total_invalidas,
    suspeitas
):

    dados = {
        "gerado_em": datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "resumo_mensal": relatorio,
        "transacoes_suspeitas": suspeitas
    }

    with open("relatorio.json", "w", encoding="utf-8") as arquivo:
        json.dump(
            dados,
            arquivo,
            ensure_ascii=False,
            indent=4,
            default=str
        )

    print("Relatório salvo com sucesso em 'relatorio.json'.")

In [106]:
salvar_json(
    relatorio,
    len(transacoes_validas),
    len(transacoes_invalidas),
    suspeitas
)

Relatório salvo com sucesso em 'relatorio.json'.
